In [3]:
import numpy as np, tensorflow as tf
from keras import layers, models
from sklearn.cluster import KMeans

# 1. Загрузка данных
print("Загрузка данных...")
states = [l.strip() for l in open('stateabbr.txt') if l.strip()]
raw = [l.strip().split(',') for l in open('plants.data', encoding='latin-1') if l.strip()]
X = np.array([np.isin(states, r[1:]).astype('float32') for r in raw])
print(f"Форма X: {X.shape}")  # покажет реальное количество строк и признаков

# 2. Кластеризация
y_train = KMeans(n_clusters=10, n_init=10, random_state=42).fit_predict(X)

# 3. Динамическая форма для CNN: подбираем прямоугольник, близкий к квадрату
n_features = X.shape[1]
h = int(np.ceil(np.sqrt(n_features)))
w = int(np.ceil(n_features / h))
X_train = np.pad(X, ((0, 0), (0, h * w - n_features)), constant_values=0).reshape(-1, h, w, 1)

# 4. Создание модели CNN
print("Создание модели...")
model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(h, w, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 5. Обучение и сохранение
print("Начинаем обучение...")
model.fit(X_train, y_train, epochs=2, batch_size=128, validation_split=0.2)
model.save('my_cnn_model.h5')
print("Модель сохранена в my_cnn_model.h5")

Загрузка данных...
Форма X: (34781, 72)


c:\Users\ismai\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1403: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\ismai\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Создание модели...
Начинаем обучение...
Epoch 1/2
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 1.0000 - loss: 2.1121 - val_accuracy: 1.0000 - val_loss: 1.9237
Epoch 2/2
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 1.0000 - loss: 1.7494 - val_accuracy: 1.0000 - val_loss: 1.5789


Модель сохранена в my_cnn_model.h5
